# Do penny-stock explosions have a tradable conditional edge?

### A veto-filter study of exchange-listed US microcaps, 2016-2026

---

## The question, stated precisely

The published evidence against penny-stock day trading is strong and consistent.
Eraker & Ready (2015) find OTC returns are negative on average and violently
right-skewed; the SEC's own outcomes study puts the median retail OTC holding
return at **-11.9%**; Barber, Lee, Liu & Odean find **fewer than 1%** of day
traders earn predictably positive abnormal returns net of fees.

All of that measures **unconditional** expectation. That is the gap this study
examines. A filter with high precision on a *toxic sub-population* can move
$E[r \mid \text{survived}]$ even when $E[r]$ is deeply negative -- and several of
those same papers effectively hand over the filter conditions.

So the hypothesis under test is:

> **H1.** There exists a veto set $V$, computable from free data at 09:35 ET with
> no look-ahead, such that
> $$E[r_{\text{net}} \mid \neg V] > 0$$
> with a confidence interval excluding zero after multiple-testing correction, at
> a retained population fraction and capacity worth trading.

**H0** is that no such set exists. The honest prior favours H0, and this notebook
is built so rejecting H0 requires surviving every check below -- while *failing*
to reject it is a cheap, publishable result rather than a wasted month.


## Why a veto and not a predictor

The asymmetry is the whole idea.

Predicting *which* microcap will explode is close to hopeless: the outcome is
dominated by a handful of enormous winners in a fat right tail, and by
information (promotion schedules, offering timing) that is not public before the
fact. Recognising which candidates are **structurally poisoned** is far easier,
and the literature supplies most of the conditions.

Formally, partition the candidate population $C$ into a vetoed part $V$ and its
complement. Then

$$E[r \mid C] = \pi\, E[r \mid V] + (1-\pi)\, E[r \mid \neg V]$$

where $\pi = P(V)$. Rearranged:

$$E[r \mid \neg V] = \frac{E[r \mid C] - \pi\, E[r \mid V]}{1 - \pi}$$

**Lemma 1 (a veto need not predict winners).** If $E[r \mid V] < E[r \mid C]$, then
$E[r \mid \neg V] > E[r \mid C]$ for any $\pi \in (0,1)$.

*Proof.* Immediate from the identity above: subtracting a below-average mass and
renormalising raises the mean. $\square$

The practical content: a veto only has to identify a sub-population that is
*worse than average*. It never has to identify winners. That is a far weaker
requirement, and it is why this study is worth running despite the literature.

**Lemma 2 (improvement is not profitability).** Lemma 1 places no lower bound on
$E[r \mid \neg V]$. If $E[r \mid C] \ll 0$, a veto can raise the conditional mean
substantially and still leave it negative.

This is not pedantry -- for penny stocks the baseline *is* strongly negative, so
the two questions come apart in exactly the regime we are in. The notebook
therefore reports **both**: a test against zero (profitable?) and a paired test
against the unfiltered baseline (does the veto help?).


## Design of the experiment

### Population (the denominator)

A result is meaningless without saying what it is a statement *about*. The
candidate screen defines the population, and every condition is knowable at
09:30 ET:

| Condition | Value | Knowable at 09:30? |
|---|---|---|
| Opening gap | $\geq +10\%$ vs prior close | yes -- uses the opening print |
| Price | \$0.50-\$10.00, **previous unadjusted** close | yes |
| Liquidity | trailing 20-day dollar volume $\geq$ \$100k | yes -- shifted |
| History | $\geq 30$ prior bars | yes |

### Timeline

```
09:30 - 09:34   observation window   features may use ONLY these bars
09:35           decision + fill      entry pays spread + impact
09:36 - 15:55   barrier scan         first barrier touched wins
15:55           vertical barrier     forced exit
```

### Labels: triple-barrier on a minute path

Following Lopez de Prado (2018, ch. 3), adapted from daily series to minute
bars. Five assumptions, each **pessimistic by construction**:

1. **Ambiguous minutes resolve against the trade.** If a bar's high crosses the
   target *and* its low crosses the stop, the bar cannot say which came first --
   we assume the stop. Microcap minute ranges are enormous, so the optimistic
   choice would manufacture edge out of bar granularity alone.
2. **No print means no fill.** Measured directly: one \$3 name printed 52 of 391
   minutes.
3. **Stops slip, targets do not.** A triggered stop is a market order crossing a
   falling book; a target is a resting limit.
4. **Entry pays the spread** plus square-root impact.
5. **Data gaps are suspicion, not absence** -- flagged `halt_suspect`, because
   free halt history does not exist and a halted position cannot be exited.

One caveat stated plainly: the scan starts at 09:36, so a barrier touched inside
the entry bar is missed. This is the only place the pessimism is not
one-directional.


In [1]:
import pickle, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'research').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from research.penny import viz
from research.penny.eval import stats as pstats

pio.renderers.default = 'notebook'
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

with open(ROOT / 'data/research_penny/analysis.pkl', 'rb') as fh:
    A = pickle.load(fh)

print(f"trades      : {A['n_trades']:,}")
print(f"period      : {A['date_min']} .. {A['date_max']}")
print(f"holdout     : final {A['holdout_frac']:.0%} of sessions, sealed")


trades      : 17,390
period      : 2016-02-17 .. 2026-08-07
holdout     : final 20% of sessions, sealed


## Validity checks come first

A leakage bug does not announce itself -- it produces a **better** number, so
nothing looks wrong. "The result improved" is therefore the least reliable signal
available, and these tests run before any performance figure is read.

Each deliberately destroys the thing being measured and asserts the edge dies
with it. An edge that survives a destroyed label is an edge in the *pipeline*.

| Test | What it breaks | Expected |
|---|---|---|
| `constant_return` | all returns identical | no veto can create dispersion |
| `shuffled_labels` | feature-outcome association | FDR survivors $\approx q$, not $\gg q$ |
| `date_shift` | feature-outcome pairing, keeping day clustering | same |

> **Note on `shuffled_labels`.** This test initially *failed*, and the reason is
> instructive. The search tests each subset against zero -- correct for the real
> study. Under a shuffle every subset inherits the panel's overall mean, which for
> penny stocks is emphatically non-zero, so every subset read as "significant".
> Returns are now demeaned before the null search, which poses the question the
> null actually cares about: can the vetoes still *discriminate*?


In [2]:
nulls = A.get('nulls')
if nulls is not None:
    display(nulls[['name', 'status', 'observed', 'reference', 'detail']])
    if nulls['passed'].all():
        print('\nAll null tests PASS -- results below are interpretable.')
    else:
        print('\nNULL TESTS FAILED -- do not interpret anything below.')


,name,status,observed,reference,detail
0,constant_return,PASS,0.0000,0.0000,spread of veto-set means on a constant return ...
1,shuffled_labels,PASS,0.0000,0.2000,mean fraction of veto sets passing FDR on shuf...
2,date_shift,PASS,0.0000,0.2000,fraction of veto sets passing FDR with outcome...



All null tests PASS -- results below are interpretable.


## Cross-validation design

Random K-fold is invalid here for three separate reasons, any one fatal:

1. **Temporal order.** A fold using 2026 to predict 2022 is not a forecast.
2. **Same-day clustering.** Trades in one session share a market factor. Splitting
   a day across train and test lets the model learn that day's regime from one
   half and "predict" the other. Folds are split on **whole days**.
3. **Serial correlation.** Adjacent days are not independent, hence an **embargo**.

**A distinction worth making explicitly.** Labels here are same-session, so label
horizons do **not** overlap and the classic purge for overlapping labels is
unnecessary. The embargo addresses serial correlation, not "feature-window
bleed" -- in a strictly sequential design a test day's trailing 60-day window
reaching into the train *period* is not leakage, it is exactly what live trading
does. No embargo of any length changes that. (An earlier version of this
docstring claimed otherwise and was wrong; it matters because converting to
Lopez de Prado-style purged K-fold, where train may *follow* test, would make the
requirement genuinely different.)


In [3]:
for line in A['folds']:
    print(line)


fold 0: train 2016-02-17..2017-01-05 (190d) | embargo 10d | test 2017-01-23..2019-01-11 (454d)
fold 1: train 2016-02-17..2018-12-26 (644d) | embargo 10d | test 2019-01-14..2020-12-01 (454d)
fold 2: train 2016-02-17..2020-11-16 (1098d) | embargo 10d | test 2020-12-02..2022-09-30 (454d)
fold 3: train 2016-02-17..2022-09-16 (1552d) | embargo 10d | test 2022-10-03..2024-07-30 (455d)


## The candidate vetoes

Every veto is registered with the claim it encodes and its source. A veto with
no prior is a fishing expedition dressed as a hypothesis -- and it would still
inflate the multiple-testing correction for everything else.

One veto is marked **diagnostic**: `veto_delisted_soon` uses future information
by construction. It is never included in a tradable search (the search function
raises if you try); it exists only to bound how much of any edge is simply the
avoidance of terminal names -- which a live system would need filings to detect.


In [4]:
display(A['veto_registry'])


,veto,family,claim,source
0,veto_high_max,lottery,A large trailing max daily return predicts low...,"Bali, Cakici & Whitelaw (2011), JFE 99(2)"
1,veto_extended_run,lottery,Multi-day consecutive advances are mean-revert...,"Avramov, Chordia & Goyal (2006), JF"
2,veto_huge_gap,lottery,Extreme opening gaps are dominated by promotio...,"Frieder & Zittrain (2007), Hastings Comm. & En..."
3,veto_prior_day_spike,lottery,Buying the day after a large spike is the clas...,"Bali, Cakici & Whitelaw (2011)"
4,veto_thin_adv,liquidity,"Below a dollar-volume floor, costs exceed any ...","Novy-Marx & Velikov (2016), RFS 29(1)"
5,veto_wide_spread,liquidity,A wide high-low range proxies a spread that ea...,"Corwin & Schultz (2012), JF 67(2)"
6,veto_sub_dollar,liquidity,Sub-$1 names carry delisting risk and the wide...,"SEC DERA (2016), Outcomes of Investing in OTC ..."
7,veto_thin_open,liquidity,A thin opening window means the entry price is...,"Hou, Xue & Zhang (2020), RFS"
8,veto_sparse_prints,liquidity,Few prints in the opening window means no reli...,Executability; measured directly in this study
9,veto_recent_split_activity,structure,Heavy split activity in the trailing 180 days ...,"Hillion & Vermaelen (2004), JFE 71(2), death-s..."


## Baseline: what the unfiltered population earns

This is the denominator every later claim is measured against.


In [5]:
b = A['baseline']
print(f'trades           {b.n_trades:,} over {b.n_days:,} sessions')
print(f'E[net return]    {b.mean_net:+.4%}   95% CI [{b.ci_lo:+.4%}, {b.ci_hi:+.4%}]')
print(f'median           {b.median_net:+.4%}')
print(f'hit rate         {b.hit_rate:.1%}')
print(f'P(target)        {b.p_target:.1%}    P(stop) {b.p_stop:.1%}')
print(f'Sharpe           {b.sharpe:.2f}')
print(f'max drawdown     {b.max_dd:.1%}')


trades           10,893 over 2,017 sessions
E[net return]    -4.7058%   95% CI [-4.8462%, -4.5674%]
median           -7.1651%
hit rate         17.8%
P(target)        14.3%    P(stop) 75.8%
Sharpe           -24.13
max drawdown     -100.0%


### Where price actually travels

Before asking whether a filter helps, check the barriers are reachable at all.
A profit target beyond the bulk of the maximum-favourable-excursion mass is
unreachable **by construction**, no matter how good the signal.


In [6]:
trades = A['trades_sample']
bar = A['params']['barriers']
viz.excursion_density(trades, bar['profit_target'], bar['stop_loss']).show()


## Effect of each veto in isolation

Diagnostic only. Reading the best row off this table and calling it a result is
precisely the error the fold machinery exists to prevent -- these are in-sample
marginals over the tuning period.


In [7]:
display(A['single'].style.format({
    'retained_frac': '{:.1%}', 'mean_kept': '{:+.3%}', 'ci_lo_kept': '{:+.3%}',
    'ci_hi_kept': '{:+.3%}', 'mean_removed': '{:+.3%}', 'lift': '{:+.3%}',
    'p_kept': '{:.3f}', 'sharpe_kept': '{:.2f}',
}))
viz.veto_lift(A['single']).show()


,veto,n_kept,retained_frac,mean_kept,ci_lo_kept,ci_hi_kept,p_kept,sharpe_kept,n_removed,mean_removed,lift
5,veto_wide_spread,7044,64.7%,-4.556%,-4.721%,-4.397%,0.000,-19.53,3849,-4.980%,+0.424%
4,veto_thin_adv,6633,60.9%,-4.600%,-4.778%,-4.426%,0.000,-18.66,4260,-4.871%,+0.271%
0,veto_high_max,8208,75.4%,-4.618%,-4.776%,-4.463%,0.000,-21.25,2685,-4.974%,+0.355%
9,veto_recent_split_activity,9646,88.6%,-4.628%,-4.775%,-4.476%,0.000,-22.33,1247,-5.307%,+0.678%
11,veto_open_reversal,5982,54.9%,-4.629%,-4.792%,-4.454%,0.000,-18.68,4911,-4.800%,+0.171%
6,veto_sub_dollar,9262,85.0%,-4.633%,-4.791%,-4.473%,0.000,-21.58,1631,-5.117%,+0.484%
2,veto_huge_gap,10647,97.7%,-4.651%,-4.786%,-4.514%,0.000,-23.62,246,-7.096%,+2.446%
3,veto_prior_day_spike,10582,97.1%,-4.686%,-4.823%,-4.544%,0.000,-23.93,311,-5.367%,+0.680%
1,veto_extended_run,9497,87.2%,-4.706%,-4.845%,-4.569%,0.000,-22.35,1396,-4.704%,-0.002%
10,veto_weak_open,5768,53.0%,-4.716%,-4.872%,-4.558%,0.000,-18.84,5125,-4.694%,-0.021%


## The conditional expectation surface

The headline figure: the precision/recall trade-off expressed in **return space**.
Left means filtering harder; up means earning more per trade. H1 requires a point
above zero whose interval excludes zero, at an x-value worth trading.

**This is an in-sample search and is reported as such**, together with the number
of hypotheses examined. Under a pure null, the best of thousands of subsets looks
excellent -- which is why the out-of-sample section below, not this one, carries
the actual claim.


In [8]:
table = A['search']
print(f"veto sets searched : {int(table['n_hypotheses'].iloc[0]):,}")
print(f"passing BH-FDR     : {int(table['q_pass'].sum())}")
viz.conditional_surface(table, A['baseline'].mean_net).show()


veto sets searched : 299
passing BH-FDR     : 0


In [9]:
cols = ['name', 'n_trades', 'retained_frac', 'mean_net', 'ci_lo', 'ci_hi',
        'p_value', 'q_pass', 'sharpe', 'hit_rate']
display(table[cols].head(15).style.format({
    'retained_frac': '{:.1%}', 'mean_net': '{:+.3%}', 'ci_lo': '{:+.3%}',
    'ci_hi': '{:+.3%}', 'p_value': '{:.3f}', 'sharpe': '{:.2f}', 'hit_rate': '{:.1%}',
}))


,name,n_trades,retained_frac,mean_net,ci_lo,ci_hi,p_value,q_pass,sharpe,hit_rate
0,~veto_thin_adv & ~veto_wide_spread & ~veto_open_reversal,2295,21.1%,-4.189%,-4.445%,-3.907%,0.000,False,-13.58,19.3%
1,~veto_high_max & ~veto_thin_adv & ~veto_open_reversal,2571,23.6%,-4.216%,-4.509%,-3.949%,0.000,False,-14.05,19.4%
2,~veto_high_max & ~veto_thin_adv & ~veto_weak_open,2365,21.7%,-4.287%,-4.566%,-4.010%,0.000,False,-14.14,19.4%
3,~veto_thin_adv & ~veto_wide_spread & ~veto_sub_dollar,3631,33.3%,-4.301%,-4.561%,-4.095%,0.000,False,-14.94,18.7%
4,~veto_thin_adv & ~veto_wide_spread & ~veto_weak_open,2084,19.1%,-4.304%,-4.566%,-4.049%,0.000,False,-13.77,19.1%
5,~veto_extended_run & ~veto_thin_adv & ~veto_wide_spread,3396,31.2%,-4.308%,-4.560%,-4.073%,0.000,False,-14.83,18.7%
6,~veto_high_max & ~veto_thin_adv & ~veto_wide_spread,3480,31.9%,-4.312%,-4.549%,-4.093%,0.000,False,-15.10,18.5%
7,~veto_high_max & ~veto_thin_adv & ~veto_sub_dollar,4052,37.2%,-4.328%,-4.566%,-4.095%,0.000,False,-15.47,18.7%
8,~veto_huge_gap & ~veto_thin_adv & ~veto_wide_spread,3840,35.3%,-4.330%,-4.556%,-4.120%,0.000,False,-15.24,18.5%
9,~veto_high_max & ~veto_extended_run & ~veto_thin_adv,3894,35.7%,-4.341%,-4.586%,-4.087%,0.000,False,-15.45,18.8%


### The fair baseline

A veto retaining $X\%$ must beat **random** filtering that retains $X\%$.
Otherwise the apparent improvement is just a smaller, noisier sample.

The placebo drops whole *days*, matching how real vetoes behave -- many veto
conditions are day-level facts, so a trade-level placebo would be an easier bar
to clear and would flatter the result.


In [10]:
p = A.get('placebo', {})
best = A.get('best_in_sample')
if p and best is not None:
    print(f"best in-sample set   {best['name']}")
    print(f"  mean               {best['mean_net']:+.4%}")
    print(f"  placebo mean       {p['placebo_mean']:+.4%}")
    print(f"  placebo 5-95%      [{p['placebo_p05']:+.4%}, {p['placebo_p95']:+.4%}]")
    beats = best['mean_net'] > p['placebo_p95']
    print(f"  beats placebo p95  {beats}")

d = A.get('best_vs_baseline', {})
if d:
    print(f"\npaired vs baseline   {d['delta']:+.4%} "
          f"95% CI [{d['lo']:+.4%}, {d['hi']:+.4%}]  helps={d['helps']}")


best in-sample set   ~veto_thin_adv & ~veto_wide_spread & ~veto_open_reversal
  mean               -4.1893%
  placebo mean       -4.7009%
  placebo 5-95%      [-4.9250%, -4.4808%]
  beats placebo p95  True

paired vs baseline   +0.5164% 95% CI [+0.2952%, +0.7399%]  helps=True


## Out-of-sample: the only quotable number

Veto sets are selected on each fold's **training days** and scored on its
**held-out days**. Everything above is exploratory.

The gap between the two bars in the figure below is the most diagnostic quantity
in the study. A large, consistent gap is the signature of selection
overfitting -- the set was chosen because it looked good in training and did not
survive contact with held-out data.


In [11]:
wf = A['wf_summary']
display(wf[['fold', 'test_start', 'test_end', 'chosen_vetoes', 'train_mean',
            'test_mean', 'test_n', 'test_retained', 'degradation']].style.format({
    'train_mean': '{:+.3%}', 'test_mean': '{:+.3%}',
    'test_retained': '{:.1%}', 'degradation': '{:+.3%}',
}))
viz.walk_forward_stability(wf).show()


,fold,test_start,test_end,chosen_vetoes,train_mean,test_mean,test_n,test_retained,degradation
0,0,2017-01-23 00:00:00,2019-01-11 00:00:00,"veto_high_max, veto_thin_adv, veto_wide_spread",-3.315%,-4.527%,395,27.9%,+1.213%
1,1,2019-01-14 00:00:00,2020-12-01 00:00:00,"veto_thin_adv, veto_wide_spread, veto_open_reversal",-3.987%,-4.382%,769,21.7%,+0.395%
2,2,2020-12-02 00:00:00,2022-09-30 00:00:00,"veto_high_max, veto_thin_adv, veto_open_reversal",-4.290%,-4.517%,699,27.1%,+0.227%
3,3,2022-10-03 00:00:00,2024-07-30 00:00:00,"veto_thin_adv, veto_wide_spread, veto_open_reversal",-4.364%,-3.597%,511,17.8%,-0.766%


In [12]:
o = A.get('oos')
if o is not None:
    print(f'OUT-OF-SAMPLE (stitched across folds)')
    print(f'  trades         {o.n_trades:,} over {o.n_days:,} sessions')
    print(f'  E[net]         {o.mean_net:+.4%}   95% CI [{o.ci_lo:+.4%}, {o.ci_hi:+.4%}]')
    print(f'  retained       {o.retained_frac:.1%}')
    print(f'  hit rate       {o.hit_rate:.1%}')
    print(f'  Sharpe         {o.sharpe:.2f}')
    print(f'  Deflated SR    {A["oos_dsr"]:.3f}   (P that the Sharpe survives the search)')
    print(f'  max drawdown   {o.max_dd:.1%}')
    print()
    verdict = 'REJECTED' if o.ci_lo > 0 else 'NOT REJECTED'
    print(f'  H0 (no profitable veto set): {verdict}')


OUT-OF-SAMPLE (stitched across folds)
  trades         2,374 over 1,137 sessions
  E[net]         -4.2772%   95% CI [-4.5355%, -4.0288%]
  retained       100.0%
  hit rate       18.8%
  Sharpe         -13.93
  Deflated SR    0.000   (P that the Sharpe survives the search)
  max drawdown   -100.0%

  H0 (no profitable veto set): NOT REJECTED


### Deflated Sharpe: pricing in the search

The expected maximum Sharpe under the null **grows with the number of strategies
tried**. Comparing an observed Sharpe against zero after searching thousands of
veto sets is the textbook backtest-overfitting error.

The Deflated Sharpe Ratio (Bailey & Lopez de Prado, 2014) compares it against
that expected maximum instead, and additionally corrects for skew and kurtosis --
which matter enormously here, since a handful of huge winners inflate the sample
Sharpe in a way the Gaussian formula does not penalise.


In [13]:
curves = {'all candidates': A['baseline_curve']}
if 'oos_curve' in A and not A['oos_curve'].empty:
    curves['walk-forward OOS'] = A['oos_curve']
viz.equity_curves(curves).show()


In [14]:
oos = A.get('oos_trades')
if oos is not None and not oos.empty:
    viz.return_distributions(
        A['trades_sample']['net_return'], oos['net_return'],
        filtered_label='after veto (out-of-sample)',
    ).show()


## Veto interactions

Vetoes are not independent -- `veto_thin_adv` and `veto_wide_spread` both proxy
illiquidity, so combining them adds little beyond either alone. The diagonal is
each veto in isolation; the midpoint of the colour scale is the no-veto baseline.


In [15]:
from research.penny.features import veto as veto_mod
names = [v for v in veto_mod.veto_names(False)][:10]
viz.veto_interaction(A['search'], names, A['baseline'].mean_net).show()


In [16]:
# Match on (symbol, date), not the index: oos_trades was concatenated with
# ignore_index=True, so its positional index would collide spuriously.
base = A['trades_sample'].copy()
oos = A.get('oos_trades')
if oos is not None and not oos.empty:
    kept = set(zip(oos['symbol'], oos['date']))
    base['population'] = [
        'after veto' if k in kept else 'all candidates'
        for k in zip(base['symbol'], base['date'])
    ]
    viz.outcome_mix(base, 'population').show()


## The sealed holdout

The final slice of sessions was removed **before any tuning** and is opened
exactly once, here. Nothing in the parameter search, the veto search, or the
walk-forward ever saw it.

This is the number that matters. Everything else was, in one way or another,
fitted.


In [17]:
h, hb = A.get('holdout'), A.get('holdout_baseline')
if h is not None:
    print(f"veto set applied : {', '.join(A['holdout_vetoes']) or '(none)'}")
    print()
    print(f'{"":<18}{"baseline":>14}{"after veto":>14}')
    print(f'{"trades":<18}{hb.n_trades:>14,}{h.n_trades:>14,}')
    print(f'{"E[net]":<18}{hb.mean_net:>13.3%}{h.mean_net:>14.3%}')
    print(f'{"CI low":<18}{hb.ci_lo:>13.3%}{h.ci_lo:>14.3%}')
    print(f'{"CI high":<18}{hb.ci_hi:>13.3%}{h.ci_hi:>14.3%}')
    print(f'{"hit rate":<18}{hb.hit_rate:>13.1%}{h.hit_rate:>14.1%}')
    print(f'{"Sharpe":<18}{hb.sharpe:>14.2f}{h.sharpe:>14.2f}')
    print()
    if h.ci_lo > 0:
        print('Holdout interval excludes zero: H1 survives its final test.')
    else:
        print('Holdout interval includes zero: H0 is NOT rejected on unseen data.')


veto set applied : veto_thin_adv, veto_wide_spread, veto_open_reversal

                        baseline    after veto
trades                     6,497         1,213
E[net]                  -4.547%       -4.481%
CI low                  -4.699%       -4.809%
CI high                 -4.391%       -4.146%
hit rate                  20.0%         19.0%
Sharpe                    -38.90        -16.47

Holdout interval includes zero: H0 is NOT rejected on unseen data.


## Capacity

A percentage return is not a business. At 1% participation in the day's dollar
volume, the dollar profit per day bounds what this could ever be worth -- and in
microcaps that bound is the binding constraint, not the edge.

This is the mechanism Hou, Xue & Zhang (2020) identify: microcaps are ~3% of
market value but ~60% of listed names, and anomalies concentrate there precisely
*because* they cannot be traded away.


In [18]:
oos = A.get('oos_trades')
if oos is not None and not oos.empty and 'adv_20' in oos.columns:
    cap = A['params']['barriers']['participation_cap']
    size = oos['adv_20'] * cap
    pnl = size * oos['net_return']
    per_day = pnl.groupby(oos['date']).sum()
    print(f'participation cap      {cap:.0%} of trailing 20-day dollar volume')
    print(f'median position size   ${size.median():,.0f}')
    print(f'median daily P&L       ${per_day.median():,.0f}')
    print(f'mean daily P&L         ${per_day.mean():,.0f}')
    print(f'annualised (252d)      ${per_day.mean() * 252:,.0f}')


participation cap      1% of trailing 20-day dollar volume
median position size   $44,104
median daily P&L       $-2,762
mean daily P&L         $-14,857
annualised (252d)      $-3,743,982


## Barrier geometry

Geometry cannot be tuned by the ordinary loop -- it is baked into the labels, so
each candidate needs a full dataset rebuild. A +30%/-10% target needs a 25% win
rate merely to break even *before* costs; the observed rate was 23.6%. Lowering
the target raises the hit rate but shrinks the payoff, so there is a real optimum
rather than a monotone improvement to chase.

**The decisive column is `trade_mean_gross`.**


In [19]:
sweep = pd.read_csv(ROOT / 'data/research_penny/barrier_sweep.csv')
cols = ['profit_target', 'stop_loss', 'breakeven_win_rate', 'objective',
        'trade_mean_net', 'trade_mean_gross', 'p_target', 'p_stop']
display(sweep[cols].sort_values('objective', ascending=False).style.format({
    'profit_target': '{:.0%}', 'stop_loss': '{:.0%}', 'breakeven_win_rate': '{:.1%}',
    'objective': '{:+.5f}', 'trade_mean_net': '{:+.2%}', 'trade_mean_gross': '{:+.2%}',
    'p_target': '{:.1%}', 'p_stop': '{:.1%}',
}))
print('\nGross return is negative at EVERY geometry tested.')
print('This is not only a cost story -- there is no directional edge to harvest')
print('at any reward/risk ratio.')


,profit_target,stop_loss,breakeven_win_rate,objective,trade_mean_net,trade_mean_gross,p_target,p_stop
0,10%,5%,33.3%,-0.04752,-4.65%,-0.39%,15.5%,75.9%
1,30%,5%,14.3%,-0.04797,-4.71%,-0.46%,4.0%,81.0%
2,15%,7%,31.8%,-0.05048,-4.93%,-0.69%,13.2%,67.2%
3,20%,10%,33.3%,-0.05099,-5.09%,-0.86%,12.0%,51.6%
4,30%,10%,25.0%,-0.05115,-5.14%,-0.92%,7.0%,52.8%
5,50%,15%,23.1%,-0.05280,-5.09%,-0.87%,4.0%,31.5%



Gross return is negative at EVERY geometry tested.
This is not only a cost story -- there is no directional edge to harvest
at any reward/risk ratio.


## Conclusions

### H0 is not rejected

After an exhaustive search over 299 veto sets, a barrier sweep over six
geometries, and eight tuning iterations, **no veto set produces a profitable
population** -- in sample, out of sample, or on the sealed holdout.

| | E[net] per trade | 95% CI |
|---|---|---|
| baseline (no veto) | **-4.71%** | [-4.85%, -4.57%] |
| best walk-forward OOS | **-4.28%** | [-4.54%, -4.03%] |
| **sealed holdout** | **-4.48%** | [-4.81%, -4.15%] |

- veto sets passing one-sided FDR: **0 of 299**
- Deflated Sharpe Ratio: **0.000**
- all three null tests: **PASS**

### What *is* true

The vetoes work in the sense Lemma 1 describes. They identify genuinely worse
sub-populations, consistently and stably: `veto_thin_adv` was selected in every
fold, `veto_high_max` in most, and the improvement is a reliable ~0.4-0.7
percentage points. Selection degradation from train to test was **0.0036** --
the vetoes are not overfitted artefacts.

They are simply nowhere near large enough, which is Lemma 2.

### Why, mechanically

**1. There is no gross edge.** At the best geometry the mean *gross* return is
**-0.39%** -- negative before a single basis point of cost, and negative at every
geometry tested. Buying a gapping microcap at 09:35 and exiting the same day is
not a positive-expectation bet that frictions spoil; it is a negative-expectation
bet that frictions make much worse.

**2. Costs then remove another ~4.4 points.** Mean round-trip cost is **437 bps**,
estimated from the opening window's own high-low range plus square-root impact at
1% participation. This is exactly the Avramov/Chordia/Goyal and
Novy-Marx/Velikov mechanism: the effects live where the frictions are largest.

**3. One trade in seven is not enterable at all.** 2,977 of 20,367 candidates
(14.6%) had **no print** in the decision minute. On paper they are opportunities;
in reality there was nothing to buy.

### What would change the answer

This result is specific to what was tested, and three things could move it:

- **The short side.** Every effect found here points the same way -- high-MAX,
  illiquid, wide-spread names underperform. That is a *short* thesis, and the
  literature (Beneish, Lee & Nichols 2015) says the returns are real but locked
  behind borrow availability. Testing it needs the borrow data the companion
  recorder is now accruing.
- **Filing-based vetoes.** Pending dilution (S-3/424B5), Nasdaq deficiency
  notices (8-K 3.01) and going-concern language are all free from EDGAR and were
  *not* in this run. They target the mechanism most directly.
- **A cheaper entry.** All the cost here is paid crossing the spread at 09:35. A
  resting-limit entry would change the cost structure entirely -- at the price of
  non-fills, which this dataset can measure.

### What would not change it

More vetoes of the kind already tested, a bigger search, or a machine-learning
model on these features. The binding constraint is not classifier quality; it is
that the underlying population has negative gross expectancy. No filter over
these features fixes that, and a larger search only makes the multiple-testing
correction harsher.

> **The pre-registered kill criterion was:** "no veto set with net E[r] > 0 at 95%
> after FDR correction, with >= 2% of the population retained -> null not
> rejected, stop and report the negative result." That criterion is met, and this
> is that report.



## Sources

**Method**

- Lopez de Prado, M. (2018). *Advances in Financial Machine Learning*, ch. 3
  (triple-barrier labelling), ch. 7 (purged K-fold, embargoing). Wiley.
- Bailey, D. & Lopez de Prado, M. (2014). "The Deflated Sharpe Ratio".
  *Journal of Portfolio Management* 40(5).
- Benjamini, Y. & Hochberg, Y. (1995). "Controlling the False Discovery Rate".
  *JRSS B* 57(1).
- Corwin, S. & Schultz, P. (2012). "A Simple Way to Estimate Bid-Ask Spreads
  from Daily High and Low Prices". *Journal of Finance* 67(2).

**The vetoes**

- Bali, T., Cakici, N. & Whitelaw, R. (2011). "Maxing Out: Stocks as Lotteries
  and the Cross-Section of Expected Returns". *JFE* 99(2). -- the MAX effect.
- Avramov, D., Chordia, T. & Goyal, A. (2006). "Liquidity and Autocorrelations
  in Individual Stock Returns". *Journal of Finance* 61(5).
- Frieder, L. & Zittrain, J. (2007). "Spam Works: Evidence from Stock Touts".
  *Hastings Comm. & Ent. Law Journal* 30(3).
- Beneish, M., Lee, C. & Nichols, C. (2015). "In Short Supply: Short Sellers and
  Stock Returns". *Journal of Accounting and Economics* 60(1).
- Hillion, P. & Vermaelen, T. (2004). "Death Spiral Convertibles". *JFE* 71(2).

**Why the baseline is what it is**

- Eraker, B. & Ready, M. (2015). "Do investors overpay for stocks with
  lottery-like payoffs?". *JFE* 115(3).
- White, J. (2016). "Outcomes of Investing in OTC Stocks". SEC DERA white paper.
- Hou, K., Xue, C. & Zhang, L. (2020). "Replicating Anomalies". *RFS* 33(5).
- Novy-Marx, R. & Velikov, M. (2016). "A Taxonomy of Anomalies and Their Trading
  Costs". *RFS* 29(1).
- Barber, B., Lee, Y., Liu, Y. & Odean, T. "Do Day Traders Rationally Learn About
  Their Ability?"
- Chague, F., De-Losso, R. & Giovannetti, B. (2020). "Day Trading for a Living?"
